<div dir="rtl">

# کارگاه جلسه ۱۹ — منظم‌سازی در رگرسیون

این دفترچه ابزارهایی را تمرین می‌کند که در جزوه نیامده‌اند. هر بخش مستقل است. نوشتن این کدها در امتحان پرسیده نمی‌شود.

1. قاعده یک خطای استاندارد برای انتخاب آلفا
2. تمرین عملی کلاس: داده مسکن کالیفرنیا و پاسخ‌ها

</div>

<div dir="rtl">

## ۱. قاعده یک خطای استاندارد

با `GridSearchCV` خطای اعتبارسنجی هر آلفا و انحراف معیار آن در پنج بخش را می‌گیریم. بزرگ‌ترین آلفایی را انتخاب می‌کنیم که میانگین خطایش حداکثر یک خطای استاندارد از بهترین خطا بیشتر باشد.

</div>

In [1]:
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.model_selection import train_test_split, KFold, GridSearchCV
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.linear_model import Ridge
from sklearn.pipeline import make_pipeline

mpg = sns.load_dataset("mpg").dropna(subset=["horsepower"])
features = ["cylinders", "displacement", "horsepower", "weight", "acceleration", "model_year"]
X_train, X_test, y_train, y_test = train_test_split(mpg[features], mpg["mpg"],
                                                    test_size=0.2, random_state=42)
kf = KFold(n_splits=5, shuffle=True, random_state=42)

model = make_pipeline(PolynomialFeatures(3, include_bias=False), StandardScaler(), Ridge())
gs = GridSearchCV(model, {"ridge__alpha": np.logspace(-4, 3, 29)}, cv=kf,
                  scoring="neg_root_mean_squared_error").fit(X_train, y_train)

res = pd.DataFrame({"alpha": gs.cv_results_["param_ridge__alpha"].astype(float),
                    "rmse": -gs.cv_results_["mean_test_score"],
                    "se": gs.cv_results_["std_test_score"] / np.sqrt(kf.get_n_splits())})
best = res.loc[res["rmse"].idxmin()]
limit = best["rmse"] + best["se"]
one_se = res.loc[res["rmse"] <= limit, "alpha"].max()
print(f"best alpha={best['alpha']:.4f}  RMSE={best['rmse']:.3f}  SE={best['se']:.3f}")
print(f"1-SE limit={limit:.3f}  chosen alpha (largest within limit)={one_se:.4f}")

best alpha=0.0562  RMSE=2.816  SE=0.099
1-SE limit=2.916  chosen alpha (largest within limit)=10.0000


<div dir="rtl">

## ۲. تمرین عملی کلاس (۲۵ دقیقه)

داده مسکن کالیفرنیا (همان داده جلسه ۱۷) را بارگذاری کنید. برای اینکه اثر بیش‌برازش دیده شود، فقط یک نمونه تصادفی ۶۰۰ بلوکی (`random_state=42`) را به کار ببرید.

1. ویژگی‌ها: میانه درآمد، میانه سن خانه‌ها، کل اتاق‌ها، کل اتاق‌خواب‌ها، جمعیت و تعداد خانوارها؛ هدف: میانه ارزش خانه به هزار دلار. داده را با `test_size=0.2` و `random_state=42` تقسیم کنید.
2. این مدل‌ها را در Pipeline (توان‌ها، سپس استانداردسازی، سپس مدل) بسازید: خطی درجه ۱، خطی درجه ۲، خطی درجه ۳، RidgeCV درجه ۳ و LassoCV درجه ۳.
3. برای هر مدل RMSE آموزش، میانگین و انحراف معیار RMSE اعتبارسنجی متقابل ۵ بخشی و RMSE و ضریب تعیین آزمون را گزارش کنید؛ RMSE مدل پایه را هم بیاورید.
4. کدام مدل بیش‌برازش شدید دارد؟ آلفای انتخابی و تعداد ضرایب غیرصفر Lasso چیست؟
5. بهترین مدل را بر اساس **اعتبارسنجی متقابل** انتخاب کنید.
6. (اختیاری) همین کار را روی **کل** داده (حدود ۲۰ هزار بلوک) تکرار کنید. فایده منظم‌سازی چه تغییری می‌کند؟

</div>

In [2]:
# Write your solution here

<div dir="rtl">

### پاسخ تمرین

</div>

In [3]:
import warnings
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.linear_model import LinearRegression, RidgeCV, LassoCV
from sklearn.pipeline import make_pipeline
from sklearn.dummy import DummyRegressor
from sklearn.metrics import root_mean_squared_error, r2_score
warnings.filterwarnings("ignore")

path = "../data/housing.csv"
housing = pd.read_csv(path).dropna(subset=["total_bedrooms"])        # rule-based
housing["value"] = housing["median_house_value"] / 1000             # rule-based
features = ["median_income", "housing_median_age", "total_rooms",
            "total_bedrooms", "population", "households"]

def poly(d, reg):
    return make_pipeline(PolynomialFeatures(d, include_bias=False), StandardScaler(), reg)

for label, data in [("sample of 600", housing.sample(n=600, random_state=42)), ("full data", housing)]:
    X_train, X_test, y_train, y_test = train_test_split(data[features], data["value"],
                                                        test_size=0.2, random_state=42)
    kf = KFold(n_splits=5, shuffle=True, random_state=42)
    models = {"Linear d1": poly(1, LinearRegression()), "Linear d2": poly(2, LinearRegression()),
              "Linear d3": poly(3, LinearRegression()),
              "RidgeCV d3": poly(3, RidgeCV(alphas=np.logspace(-3, 4, 50))),
              "LassoCV d3": poly(3, LassoCV(cv=kf, max_iter=50000, random_state=42))}
    print(f"--- {label}: train={len(X_train)}  test={len(X_test)}")
    for name, m in models.items():
        cv = -cross_val_score(m, X_train, y_train, cv=kf, scoring="neg_root_mean_squared_error")
        m.fit(X_train, y_train)
        extra = (f"  alpha={m[-1].alpha_:.3g}  nonzero={np.sum(m[-1].coef_ != 0)}/{len(m[-1].coef_)}"
                 if hasattr(m[-1], "alpha_") else "")
        print(f"{name:11s} train={root_mean_squared_error(y_train, m.predict(X_train)):6.1f}  "
              f"CV={cv.mean():7.1f} (sd {cv.std():6.1f})  "
              f"test={root_mean_squared_error(y_test, m.predict(X_test)):6.1f}  "
              f"R2={r2_score(y_test, m.predict(X_test)):.3f}{extra}")
    base = DummyRegressor().fit(X_train, y_train)
    print(f"baseline test RMSE={root_mean_squared_error(y_test, base.predict(X_test)):.1f}")

--- sample of 600: train=480  test=120
Linear d1   train=  75.8  CV=   78.8 (sd    5.3)  test=  80.6  R2=0.518
Linear d2   train=  67.5  CV=   89.9 (sd   28.8)  test=  74.7  R2=0.585
Linear d3   train=  55.2  CV=  898.2 (sd 1557.2)  test= 102.7  R2=0.217


RidgeCV d3  train=  65.5  CV=  120.8 (sd   78.9)  test=  73.3  R2=0.601  alpha=1.93  nonzero=83/83


LassoCV d3  train=  71.1  CV=   75.2 (sd    7.1)  test=  76.7  R2=0.563  alpha=1.19  nonzero=12/83
baseline test RMSE=116.1
--- full data: train=16346  test=4087
Linear d1   train=  75.8  CV=   76.0 (sd    1.0)  test=  76.6  R2=0.571
Linear d2   train=  70.6  CV=   74.2 (sd    5.6)  test=  71.5  R2=0.626


Linear d3   train=  65.8  CV=   70.4 (sd    3.7)  test=  68.8  R2=0.654


RidgeCV d3  train=  66.7  CV=   71.5 (sd    4.8)  test=  68.7  R2=0.655  alpha=10  nonzero=83/83


LassoCV d3  train=  69.7  CV=   72.0 (sd    4.4)  test=  70.9  R2=0.633  alpha=0.423  nonzero=21/83
baseline test RMSE=116.9


<div dir="rtl">

### پاسخ‌های کلیدی

- نمونه ۶۰۰ بلوکی: ۴۸۰ آموزش و ۱۲۰ آزمون. RMSE مدل پایه روی آزمون حدود **۱۱۶** هزار دلار است.
- **خطی درجه ۳** (۸۳ ویژگی) بیش‌برازش شدید دارد: RMSE آموزش حدود ۵۵ ولی RMSE اعتبارسنجی حدود **۹۰۰** با انحراف معیار بسیار بزرگ؛ در یکی از بخش‌ها پیش‌بینی‌ها منفجر می‌شوند.
- مدل **RidgeCV درجه ۳** (آلفا حدود ۱.۹) خطای اعتبارسنجی را به حدود ۱۲۱ می‌رساند اما هنوز در یک بخش ناپایدار است. مدل **LassoCV درجه ۳** (آلفا حدود ۱.۲) فقط **۱۲** ویژگی از ۸۳ را نگه می‌دارد و بهترین خطای اعتبارسنجی (حدود **۷۵**) را دارد؛ کمی بهتر از خطی درجه ۱ (حدود ۷۹).
- انتخاب بر اساس اعتبارسنجی: **LassoCV درجه ۳**. روی آزمون RMSE حدود ۷۷ و ضریب تعیین حدود ۰.۵۶.
- روی کل داده (حدود ۱۶ هزار نمونه آموزش) خطی درجه ۳ دیگر بیش‌برازش ندارد (اعتبارسنجی حدود ۷۰) و منظم‌سازی تقریباً بی‌اثر است: با داده زیاد، **واریانس** کم است و جریمه چیز زیادی برای کم کردن ندارد.

</div>